In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
import torch
from scipy.signal import find_peaks
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [2]:
import matplotlib
matplotlib.use("Agg")   # non-interactive backend, avoids the inline backend bug

import sys
sys.path.append("/data1/yashvi_bhuva/anemia/PPG_BP/papagei-foundation-model")

import pyPPG.preproc as PP
from dotmap import DotMap
from linearprobing.utils import resample_batch_signal, load_model_without_module_prefix
from models.resnet import ResNet1DMoE

In [3]:
sys.path.append("/data1/yashvi_bhuva/anemia/PPG_BP/papagei-foundation-model")

import pyPPG.preproc as PP
from dotmap import DotMap
from linearprobing.utils import resample_batch_signal, load_model_without_module_prefix
from models.resnet import ResNet1DMoE


In [10]:
RAW_DIR = "0_subject"              # folder of raw .txt files, e.g. "2_1.txt"
FS_ORIGINAL = 1000                 # PPG-BP native sampling rate — confirm
FS_TARGET_PAPAGEI = 125            # PaPaGei-S expected input rate
WEIGHTS_PATH = "papagei-foundation-model/weights/papagei_s.pt"
METADATA_PATH = "PPG-BP dataset.xlsx"

fname_pattern = re.compile(r"^(\d+)_(\d+)")


# ============================================================
meta = pd.read_excel(METADATA_PATH, sheet_name="cardiovascular dataset", header=1)
meta["subject_ID"] = meta["subject_ID"].astype(int)
meta = meta.set_index("subject_ID")


def get_peaks(ppg, fs):
    peaks, _ = find_peaks(ppg, distance=int(0.4 * fs))
    return peaks

def heart_rate_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    if len(peaks) < 2:
        return np.nan
    ppi = np.diff(peaks) / fs
    return 60 / np.mean(ppi)

def mean_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(ppg[peaks]) if len(peaks) > 0 else np.nan

def std_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.std(ppg[peaks]) if len(peaks) > 0 else np.nan

def mean_peak_distance_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(np.diff(peaks)) if len(peaks) >= 2 else np.nan

def extract_features(ppg, vpg, apg, fs):
    features = {
        "mean_ppg": np.mean(ppg), "std_ppg": np.std(ppg),
        "skew_ppg": stats.skew(ppg), "kurtosis_ppg": stats.kurtosis(ppg),
        "rms_ppg": np.sqrt(np.mean(ppg**2)), "max_ppg": np.max(ppg),
        "min_ppg": np.min(ppg), "range_ppg": np.ptp(ppg),
        "heart_rate": heart_rate_feature(ppg, fs),
        "mean_peak_amp": mean_peak_amplitude_feature(ppg, fs),
        "std_peak_amp": std_peak_amplitude_feature(ppg, fs),
        "mean_peak_distance": mean_peak_distance_feature(ppg, fs),
        "max_vpg": np.max(vpg), "min_vpg": np.min(vpg),
        "mean_vpg": np.mean(vpg), "std_vpg": np.std(vpg),
        "max_apg": np.max(apg), "min_apg": np.min(apg),
        "mean_apg": np.mean(apg), "std_apg": np.std(apg),
        "dominant_freq": np.nan, "spectral_energy": np.sum(ppg**2),
    }
    freqs = np.fft.rfftfreq(len(ppg), d=1/fs)
    fft_mag = np.abs(np.fft.rfft(ppg))
    if len(fft_mag) > 1:
        features["dominant_freq"] = freqs[1:][np.argmax(fft_mag[1:])]
    return features



def preprocess_one_ppg_signal(waveform, frequency, fL=0.5, fH=12, order=4,
                                smoothing_windows={"ppg": 50, "vpg": 10, "apg": 10, "jpg": 10}):
    prep = PP.Preprocess(fL=fL, fH=fH, order=order, sm_wins=smoothing_windows)
    signal = DotMap()
    signal.v = waveform
    signal.fs = frequency
    signal.filtering = True
    ppg, ppg_d1, ppg_d2, ppg_d3 = prep.get_signals(signal)
    return ppg, ppg_d1, ppg_d2, ppg_d3


def is_signal_flat_lined_simple(sig, fs, flat_threshold=0.25, change_threshold=0.01, window_ms=20):
    sig = np.asarray(sig, dtype=float)
    sig_norm = (sig - np.mean(sig)) / (np.std(sig) + 1e-8)

    step = max(1, int(fs * window_ms / 1000))  # compare samples ~20ms apart, not adjacent
    diffs = np.abs(sig_norm[step:] - sig_norm[:-step])
    flat_fraction = np.sum(diffs < change_threshold) / len(diffs)
    return 1 if flat_fraction > flat_threshold else 0

In [11]:
# ============================================================
# SECTION 6: Load PaPaGei-S model
# ============================================================
model_config = {
    'base_filters': 32, 'kernel_size': 3, 'stride': 2, 'groups': 1,
    'n_block': 18, 'n_classes': 512, 'n_experts': 3
}
papagei_model = ResNet1DMoE(in_channels=1, **model_config)
papagei_model = load_model_without_module_prefix(papagei_model, WEIGHTS_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
papagei_model.to(device).eval()


# ============================================================
# SECTION 7: Main loop — read raw signal, filter, QC, extract
#            BOTH hand-crafted features AND PaPaGei embedding
# ============================================================
log_rows = []
feature_rows = []
n_ok, n_fail, n_flatlined = 0, 0, 0

for file in os.listdir(RAW_DIR):
    try:
        match = fname_pattern.match(file)
        if not match:
            raise ValueError(f"Could not parse subject_id/segment from '{file}'")
        subject_id, segment = int(match.group(1)), int(match.group(2))
        if subject_id not in meta.index:
            raise KeyError(f"subject_id {subject_id} not in metadata")

        raw_ppg = np.loadtxt(os.path.join(RAW_DIR, file))

        # -- filter + derivatives (PaPaGei's exact preprocessing) --
        ppg_filt, vpg_filt, apg_filt, _ = preprocess_one_ppg_signal(
            waveform=raw_ppg, frequency=FS_ORIGINAL
        )

        # -- quality gate --
        if is_signal_flat_lined_simple(ppg_filt, fs=FS_ORIGINAL):
            n_flatlined += 1
            log_rows.append({"file": file, "status": "flatlined", "error": ""})
            continue

        # -- hand-crafted features (on the same filtered signal) --
        feature_row = extract_features(ppg_filt, vpg_filt, apg_filt, FS_ORIGINAL)

        # -- PaPaGei embedding --
        resampled = resample_batch_signal(
            ppg_filt[np.newaxis, :], fs_original=FS_ORIGINAL,
            fs_target=FS_TARGET_PAPAGEI, axis=-1
        )
        signal_tensor = torch.Tensor(resampled).unsqueeze(dim=1).to(device)
        with torch.inference_mode():
            outputs = papagei_model(signal_tensor)
            embedding = outputs[0].cpu().numpy().squeeze()
        for i, v in enumerate(embedding):
            feature_row[f"papagei_{i}"] = v

        # -- labels / demographics --
        label_row = meta.loc[subject_id]
        feature_row["SBP"] = label_row["Systolic Blood Pressure(mmHg)"]
        feature_row["DBP"] = label_row["Diastolic Blood Pressure(mmHg)"]
        feature_row["Age"] = label_row["Age(year)"]
        feature_row["BMI"] = label_row["BMI(kg/m^2)"]
        feature_row["Hypertension"] = label_row["Hypertension"]

        feature_row["file"] = file
        feature_row["subject_id"] = subject_id
        feature_row["segment"] = segment

        feature_rows.append(feature_row)
        log_rows.append({"file": file, "status": "ok", "error": ""})
        n_ok += 1

    except Exception as e:
        log_rows.append({"file": file, "status": "failed", "error": str(e)})
        n_fail += 1
        print(f"[FAILED] {file}: {e}")

print(f"\nDone. {n_ok} ok, {n_flatlined} flatlined (excluded), {n_fail} failed.")
features_df = pd.DataFrame(feature_rows)
log_df = pd.DataFrame(log_rows)
print(features_df.shape)



Done. 657 ok, 0 flatlined (excluded), 0 failed.
(657, 542)


In [12]:
# ============================================================
# SECTION 8: Aggregate to subject level
# ============================================================
hand_crafted_cols = ["mean_ppg","std_ppg","skew_ppg","kurtosis_ppg","rms_ppg","max_ppg",
    "min_ppg","range_ppg","heart_rate","mean_peak_amp","std_peak_amp","mean_peak_distance",
    "max_vpg","min_vpg","mean_vpg","std_vpg","max_apg","min_apg","mean_apg","std_apg",
    "dominant_freq","spectral_energy"]
papagei_cols = [c for c in features_df.columns if c.startswith("papagei_")]

subject_df = features_df.groupby("subject_id")[hand_crafted_cols + papagei_cols].mean().reset_index()
labels = features_df.drop_duplicates("subject_id")[["subject_id","SBP","DBP","Age","BMI","Hypertension"]]
subject_df = subject_df.merge(labels, on="subject_id")
print(subject_df.shape)

(219, 540)


In [13]:

# ============================================================
# SECTION 9: Three-condition comparison
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
data = subject_df.dropna(subset=["DBP"])
y = data["DBP"].values

conditions = {
    "Demographics only":      ["BMI", "Age"],
    "Hand-crafted PPG feats": hand_crafted_cols + ["BMI", "Age"],
    "PaPaGei embeddings":     papagei_cols + ["BMI", "Age"],
}

for name, cols in conditions.items():
    X = data[cols].values
    pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    r2 = cross_val_score(pipe, X, y, cv=kf, scoring="r2")
    print(f"{name}: R² = {r2.mean():.3f} ± {r2.std():.3f}")

Demographics only: R² = 0.010 ± 0.035
Hand-crafted PPG feats: R² = 0.045 ± 0.100
PaPaGei embeddings: R² = -0.860 ± 0.779


In [14]:
from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA

# Try much stronger regularization, tuned automatically per fold
alphas = [0.1, 1, 10, 100, 1000, 10000]

# 1) Just stronger Ridge alpha, no dimensionality reduction
X = data[papagei_cols + ["BMI", "Age"]].values
pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))
r2 = cross_val_score(pipe, X, y, cv=kf, scoring="r2")
print(f"PaPaGei + RidgeCV (auto-tuned alpha): R² = {r2.mean():.3f} ± {r2.std():.3f}")

# 2) PCA down to far fewer components first, THEN Ridge
for n_comp in [5, 10, 20, 50]:
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_comp),
        RidgeCV(alphas=alphas)
    )
    r2 = cross_val_score(pipe, X, y, cv=kf, scoring="r2")
    print(f"PaPaGei PCA({n_comp}) + RidgeCV: R² = {r2.mean():.3f} ± {r2.std():.3f}")

PaPaGei + RidgeCV (auto-tuned alpha): R² = 0.017 ± 0.051
PaPaGei PCA(5) + RidgeCV: R² = 0.009 ± 0.039
PaPaGei PCA(10) + RidgeCV: R² = 0.006 ± 0.039
PaPaGei PCA(20) + RidgeCV: R² = 0.003 ± 0.034
PaPaGei PCA(50) + RidgeCV: R² = 0.023 ± 0.049
